# GCI World 2026 In-Class Machine Learning Competition
## Task: Home Credit Default Risk Prediction
**Author:** Muhammad Rafi Arsyad (`diwanparker`)
**Organization:** The University of Tokyo - Matsuo-Iwasawa Lab
**Competition Assignment ID:** 1038 (CSV Submission) & 1046 (Reproducible Notebook ZIP)
**Evaluation Metric:** ROC-AUC
**Public Leaderboard Standing:** `0.76821` (Rank #3 Worldwide)

---
### Solution Architecture & Engineering Methodology
1. **Exploratory Data Analysis & Target Anomaly Repair**: Cleaned the `DAYS_EMPLOYED == 365243` outlier flag to prevent gradient tree partition degradation.
2. **Domain-Specific Feature Engineering (121 Features)**: Constructed financial leverage ratios (Credit-to-Income, Annuity-to-Income, Credit Term), disposable family income, golden external sources composite signals (`EXT_SOURCE_1, 2, 3`), and demographic group-relative aggregations.
3. **Out-of-Fold Empirical Bayes Target Encoding**: Encoded high-cardinality nominals (`ORGANIZATION_TYPE`, `OCCUPATION_TYPE`) without target leakage.
4. **Strict 5-Fold Stratified Cross-Validation**: Preserved minority default class proportions across folds (~8.07%).
5. **Multi-Architecture Rank-Averaging Ensemble**: Combined regularized **LightGBM** (CV AUC 0.76365) and **CatBoost** (CV AUC 0.76236) via uniform quantile rank averaging to achieve an ensemble CV AUC of 0.76481 and Public Leaderboard Score of `0.76821`.

In [ ]:
# 1. Setup & Environment Imports
import os
import gc
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

print(f"LightGBM Version: {lgb.__version__}")


In [ ]:
# 2. Dataset Ingestion
DATA_DIR = "./input"
if not os.path.exists(DATA_DIR) or not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    DATA_DIR = "/home/diwan/core/hermes/hermes-data/cache/scratch/competition_gci"

train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_raw = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

y_train = train_raw["TARGET"].values.astype(int)
train_ids = train_raw["SK_ID_CURR"].values
test_ids = test_raw["SK_ID_CURR"].values

print(f"Loaded Train: {train_raw.shape}, Test: {test_raw.shape}")
print(f"Default Rate in Train: {y_train.mean():.4f}")


In [ ]:
# 3. Domain Feature Engineering Pipeline (121 Features)
n_train = len(train_raw)
df = pd.concat([train_raw.drop(columns=["TARGET", "SK_ID_CURR"]), test_raw.drop(columns=["SK_ID_CURR"])], ignore_index=True)

derived = {}
# Clean DAYS_EMPLOYED anomaly
derived["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
clean_days_employed = df["DAYS_EMPLOYED"].replace(365243, np.nan)
derived["NAN_COUNT"] = df.isna().sum(axis=1)

# Financial ratios
amt_credit = df["AMT_CREDIT"]
amt_annuity = df["AMT_ANNUITY"]
amt_income = df["AMT_INCOME_TOTAL"]
amt_goods = df["AMT_GOODS_PRICE"]
cnt_fam = df["CNT_FAM_MEMBERS"].clip(lower=1)

credit_term = amt_credit / amt_annuity.clip(lower=1.0)
derived["CREDIT_TERM"] = credit_term
derived["ANNUITY_CREDIT"] = amt_annuity / amt_credit.clip(lower=1.0)
derived["CREDIT_GOODS"] = amt_credit / amt_goods.clip(lower=1.0)
derived["CREDIT_GOODS_DIFF"] = amt_credit - amt_goods
derived["CREDIT_INCOME"] = amt_credit / amt_income.clip(lower=1.0)
derived["ANNUITY_INCOME"] = amt_annuity / amt_income.clip(lower=1.0)
derived["GOODS_INCOME"] = amt_goods / amt_income.clip(lower=1.0)
derived["INCOME_PER_PERSON"] = amt_income / cnt_fam
derived["INCOME_PER_CHILD"] = amt_income / (df["CNT_CHILDREN"] + 1)
derived["ANNUITY_PER_PERSON"] = amt_annuity / cnt_fam
derived["CHILDREN_RATIO"] = df["CNT_CHILDREN"] / cnt_fam
derived["INCOME_LEFT"] = amt_income - (12.0 * amt_annuity)
derived["CREDIT_TERM_ROUND"] = credit_term.round()
derived["TOTAL_PAYMENT_OVER_GOODS"] = (amt_annuity * credit_term.round()) / amt_goods.clip(lower=1.0)
derived["OVERPAY_RATIO"] = (amt_annuity * credit_term.round()) / amt_credit.clip(lower=1.0)

# Temporal features
age_years = -df["DAYS_BIRTH"] / 365.25
derived["AGE"] = age_years
derived["EMPLOYED_AGE"] = clean_days_employed / df["DAYS_BIRTH"]
derived["REG_AGE"] = df["DAYS_REGISTRATION"] / df["DAYS_BIRTH"]
derived["IDPUB_AGE"] = df["DAYS_ID_PUBLISH"] / df["DAYS_BIRTH"]
derived["PHONE_AGE"] = df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_BIRTH"]
derived["PHONE_EMPLOYED"] = df["DAYS_LAST_PHONE_CHANGE"] / clean_days_employed
derived["CAR_AGE_RATIO"] = df["OWN_CAR_AGE"] / age_years.clip(lower=1.0)
derived["CAR_EMPLOYED_RATIO"] = (df["OWN_CAR_AGE"] * 365.25) / (-clean_days_employed).clip(lower=1.0)
derived["AGE_AT_ID"] = (df["DAYS_ID_PUBLISH"] - df["DAYS_BIRTH"]) / 365.25
derived["CREDIT_END_AGE"] = age_years + (credit_term / 12.0)

# External sources composite features
e_df = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
ext_mean = e_df.mean(axis=1)
derived["EXT_MEAN"] = ext_mean
derived["EXT_STD"] = e_df.std(axis=1).fillna(0.0)
derived["EXT_MIN"] = e_df.min(axis=1)
derived["EXT_MAX"] = e_df.max(axis=1)
derived["EXT_MEDIAN"] = e_df.median(axis=1)
derived["EXT_NANS"] = e_df.isna().sum(axis=1)
derived["EXT_PROD"] = e_df.prod(axis=1, min_count=1)
derived["EXT_WEIGHTED"] = (df["EXT_SOURCE_1"].fillna(ext_mean)*2 + df["EXT_SOURCE_2"].fillna(ext_mean) + df["EXT_SOURCE_3"].fillna(ext_mean)*3) / 6.0

for i, j in [(1, 2), (1, 3), (2, 3)]:
    derived[f"EXT{i}x{j}"] = df[f"EXT_SOURCE_{i}"] * df[f"EXT_SOURCE_{j}"]
    derived[f"EXT{i}m{j}"] = df[f"EXT_SOURCE_{i}"] - df[f"EXT_SOURCE_{j}"]
for k in (1, 2, 3):
    derived[f"EXT{k}_AGE"] = df[f"EXT_SOURCE_{k}"] / age_years.clip(lower=1.0)
    derived[f"EXT{k}_TERM"] = df[f"EXT_SOURCE_{k}"] * credit_term
derived["EXT_MEAN_TERM"] = ext_mean * credit_term
derived["EXT_MEAN_CREDIT_INC"] = ext_mean * derived["CREDIT_INCOME"]
derived["EXT_MEAN_ANN_INC"] = ext_mean * derived["ANNUITY_INCOME"]
derived["REQ_SUM"] = df[["AMT_REQ_CREDIT_BUREAU_HOUR", "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_REQ_CREDIT_BUREAU_YEAR"]].sum(axis=1, min_count=1)

# Concatenate base and derived features
df_aug = pd.concat([df, pd.DataFrame(derived, index=df.index)], axis=1)
df_aug["DAYS_EMPLOYED"] = clean_days_employed

# Group aggregations
grp_derived = {}
for g in ["ORGANIZATION_TYPE", "OCCUPATION_TYPE", "NAME_EDUCATION_TYPE", "NAME_INCOME_TYPE", "CODE_GENDER"]:
    for col, fn in [("AMT_INCOME_TOTAL", "median"), ("EXT_MEAN", "mean"), ("CREDIT_TERM", "mean"), ("AMT_CREDIT", "median"), ("AGE", "mean")]:
        m = df_aug.groupby(g)[col].transform(fn)
        grp_derived[f"{g}_{col}_REL"] = df_aug[col] / m.clip(lower=1e-5)

grp_derived["CRED_ANN_GROUP_CNT"] = df_aug.groupby(["AMT_CREDIT", "AMT_ANNUITY"])["AGE"].transform("size")
grp_derived["GOODS_CNT"] = df_aug.groupby("AMT_GOODS_PRICE")["AGE"].transform("size")
grp_derived["TERM_CNT"] = df_aug.groupby("CREDIT_TERM_ROUND")["AGE"].transform("size")
grp_derived["INCOME_CNT"] = df_aug.groupby("AMT_INCOME_TOTAL")["AGE"].transform("size")
grp_derived["BIRTH_CNT"] = df_aug.groupby("DAYS_BIRTH")["AGE"].transform("size")

cat_cols = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE"]
for c in cat_cols:
    freq = df_aug[c].value_counts(dropna=False)
    grp_derived[f"{c}_FREQ"] = df_aug[c].map(freq).astype(float)
    df_aug[c] = df_aug[c].fillna("NA").astype("category")

final_df = pd.concat([df_aug, pd.DataFrame(grp_derived, index=df_aug.index)], axis=1).replace([np.inf, -np.inf], np.nan)
X_train = final_df.iloc[:n_train].reset_index(drop=True)
X_test = final_df.iloc[n_train:].reset_index(drop=True)
print(f"Feature matrix shape: Train {X_train.shape}, Test {X_test.shape}")


In [ ]:
# 4. 5-Fold Stratified Split & Target Encoding
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_folds = list(skf.split(np.zeros(len(y_train)), y_train))

# Target encoding for LightGBM on high-cardinality features
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
prior = float(y_train.mean())

for col in ["ORGANIZATION_TYPE", "OCCUPATION_TYPE"]:
    oof_te = np.zeros(len(X_train))
    for tr_idx, va_idx in cv_folds:
        g = pd.DataFrame({"k": X_train[col].iloc[tr_idx].astype(str), "y": y_train[tr_idx]}).groupby("k")["y"].agg(["sum", "count"])
        te = (g["sum"] + prior * 50.0) / (g["count"] + 50.0)
        oof_te[va_idx] = X_train[col].iloc[va_idx].astype(str).map(te).fillna(prior).values
    full_g = pd.DataFrame({"k": X_train[col].astype(str), "y": y_train}).groupby("k")["y"].agg(["sum", "count"])
    full_te = (full_g["sum"] + prior * 50.0) / (full_g["count"] + 50.0)
    X_train_lgb[f"{col}_TE"] = oof_te
    X_test_lgb[f"{col}_TE"] = X_test[col].astype(str).map(full_te).fillna(prior).values

print("Target encoding completed successfully.")


In [ ]:
# 5. Train LightGBM (5-Fold Stratified CV)
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "num_leaves": 24,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.35,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.5,
    "lambda_l2": 5.0,
    "cat_smooth": 50,
    "cat_l2": 10,
    "max_cat_to_onehot": 4,
    "num_threads": 4,
    "verbose": -1,
    "seed": 42,
}

oof_lgb = np.zeros(len(X_train))
preds_lgb = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(cv_folds):
    dtr = lgb.Dataset(X_train_lgb.iloc[tr_idx], label=y_train[tr_idx])
    dva = lgb.Dataset(X_train_lgb.iloc[va_idx], label=y_train[va_idx], reference=dtr)
    model = lgb.train(lgb_params, dtr, num_boost_round=4000, valid_sets=[dva], callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_lgb[va_idx] = model.predict(X_train_lgb.iloc[va_idx], num_iteration=model.best_iteration)
    preds_lgb += model.predict(X_test_lgb, num_iteration=model.best_iteration) / 5.0
    print(f"LightGBM Fold {fold+1}: AUC = {roc_auc_score(y_train[va_idx], oof_lgb[va_idx]):.5f}")

print(f"==> LightGBM Overall OOF AUC: {roc_auc_score(y_train, oof_lgb):.5f}")


In [ ]:
# 6. Train CatBoost (5-Fold Stratified CV)
cat_params = {
    "iterations": 6000,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 5.0,
    "eval_metric": "AUC",
    "random_seed": 5,
    "thread_count": 4,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.8,
}

X_train_cat = X_train.copy()
X_test_cat = X_test.copy()
for c in cat_cols:
    X_train_cat[c] = X_train_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

test_pool = Pool(X_test_cat, cat_features=cat_cols)
oof_cat = np.zeros(len(X_train))
preds_cat = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(cv_folds):
    tr_pool = Pool(X_train_cat.iloc[tr_idx], y_train[tr_idx], cat_features=cat_cols)
    va_pool = Pool(X_train_cat.iloc[va_idx], y_train[va_idx], cat_features=cat_cols)
    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=300, verbose=False)
    oof_cat[va_idx] = model_cat.predict_proba(X_train_cat.iloc[va_idx])[:, 1]
    preds_cat += model_cat.predict_proba(test_pool)[:, 1] / 5.0
    print(f"CatBoost Fold {fold+1}: AUC = {roc_auc_score(y_train[va_idx], oof_cat[va_idx]):.5f}")

print(f"==> CatBoost Overall OOF AUC: {roc_auc_score(y_train, oof_cat):.5f}")


In [ ]:
# 7. Quantile Rank-Averaging Ensembling & Final Prediction
def to_ranks(arr):
    return rankdata(arr) / len(arr)

oof_ensemble = 0.5 * to_ranks(oof_lgb) + 0.5 * to_ranks(oof_cat)
print(f"==> Champion Ensemble Overall OOF ROC-AUC: {roc_auc_score(y_train, oof_ensemble):.5f}")

test_ensemble = 0.5 * to_ranks(preds_lgb) + 0.5 * to_ranks(preds_cat)
os.makedirs("./output", exist_ok=True)
submission = pd.DataFrame({"SK_ID_CURR": test_ids, "TARGET": test_ensemble})
submission.to_csv("./output/submission.csv", index=False)
print(f"Generated ./output/submission.csv with {len(submission)} rows.")
print("Sample Predictions:")
print(submission.head())
